<a href="https://colab.research.google.com/github/harikrishna376/ai-market-sentinel-v2/blob/main/rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# 1. Install the missing dependency
!sudo apt-get update && sudo apt-get install -y zstd

# 2. Re-run the installation script
!curl -fsSL https://ollama.com/install.sh | sh

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://cli.github.com/packages stable InRelease [3,917 B]
Get:5 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [7,143 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [88.5 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-upda

In [6]:
!pip install -U langchain-text-splitters

In [15]:
import subprocess
import time
import os

# Start Ollama server in the background
# We use a log file to see if the server runs into trouble
with open("ollama_server.log", "w") as f:
    process = subprocess.Popen(
        ['ollama', 'serve'],
        stdout=f,
        stderr=f
    )

# Give it 10 seconds to initialize the API
time.sleep(10)

# Check if it's alive and pull the Smart model (4.5 Billion parameters)
!ollama pull gemma4:e4b

In [16]:
!ollama run gemma4:e4b "Say 'System Online' if you can hear me."

Thinking...
Thinking Process:

1.  **Analyze the Request:** The user asks me to "Say 'System Online' if yo
you can hear me."
2.  **Determine My Modality/State:** I am an AI text model. I do not have p
physical senses (hearing). I process text input.
3.  **Interpret the Condition:** The phrase "if you can hear me" implies an
an auditory capability that I lack.
4.  **Determine the Correct Action:** Since I cannot literally "hear" anyth
anything, the premise of the conditional statement ("if you can hear me") i
is technically false in the physical sense.
5.  **Formulate the Response (Addressing the Limitation):** I must inform t
the user that I process text, not audio, but I should still provide the des
desired text response as a helpful response, while maintaining accuracy abo
about my capabilities.

*Self-Correction/Refinement:* Simply saying "I cannot hear you" might seem 
unhelpful. The best approach is to acknowledge the limitation (audio input)
input) and confirm the desired output 

In [17]:
!pip install -qU langchain-classic

In [19]:
# 1. Install EVERYTHING (Modular 2026 Versions)
!pip install -qU langchain langchain-community langchain-ollama langchain-text-splitters langchain-classic sentence-transformers faiss-cpu pypdf

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaLLM
from langchain_classic.chains import RetrievalQA # Legacy import for RetrievalQA

# 2. LOAD & SPLIT
# Check: Is your file actually named 'resume.pdf' and uploaded to the folder icon on the left?
loader = PyPDFLoader("RESUME.pdf")
data = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = text_splitter.split_documents(data)

# 3. EMBEDDINGS (The Search Logic)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_db = FAISS.from_documents(chunks, embeddings)

# 4. LLM CONNECTION
# Ensure your Ollama server cell is still running in the background!
llm = OllamaLLM(model="gemma4:e4b")

# 5. THE RAG PIPELINE
rag_pipeline = RetrievalQA.from_chain_type(llm=llm, retriever=vector_db.as_retriever())

# 6. ASK THE RECRUITER
query = "What are the candidate's top 3 skills, and how do they fit a Data Science role?"
response = rag_pipeline.invoke(query)
print("\n--- GEMMA 4 ANALYSIS ---")
print(response['result'])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- GEMMA 4 ANALYSIS ---
Based on the context provided, the candidate's top 3 skills relevant to a Data Science role are:

1.  **Python Programming:** The candidate lists Python under Technical Strengths and has a "Programming in Python" certification. Python is the industry-standard language for data science, crucial for data manipulation, analysis, and modeling.
2.  **Machine Learning:** The candidate holds a "Machine Learning" certification from Coursera. This is the core theoretical and practical skill of advanced Data Science, allowing the candidate to build predictive models.
3.  **Cloud Computing (AWS):** The candidate has completed the "Introduction to cloud by aws" certification. This skill is vital for deploying, scaling, and operationalizing data science models in a real-world cloud environment.

***

### How these skills fit a Data Science role:

*   **Python** serves as the primary toolset for data ingestion, cleaning, and execution of algorithms.
*   **Machine Learning**

In [20]:
query = """
I am applying for a Junior Data Scientist role at a top-tier tech company.
Based ONLY on the uploaded resume:
1. What is the biggest technical 'red flag' or missing skill?
2. If you were the hiring manager, why would you REJECT this candidate?
3. What is one specific project I should build next to fix these weaknesses?
"""
response = rag_pipeline.invoke(query)
print(response['result'])

Based ONLY on the uploaded resume and targeting a Junior Data Scientist role:

***

### 1. What is the biggest technical 'red flag' or missing skill?

The biggest red flag is the **lack of demonstrable, advanced proficiency in statistical techniques and structured data manipulation.**

While the resume lists Python and includes "Machine Learning" in certifications, the projects shown ("Hospital Management System," "Virtual Try-On," "Youtube transcript summarization") focus heavily on *application development* (full-stack/image processing/web services) rather than the core data science cycle. A junior data scientist role requires deep evidence of proficiency in statistical modeling, advanced data cleaning (e.g., handling missing values, identifying data biases), and the use of core scientific libraries like **Pandas, NumPy, and advanced SQL querying**—all of which are not highlighted with the depth required for a DS role.

### 2. If you were the hiring manager, why would you REJECT this